In [ ]:
import torch
import numpy as np
import tempfile
import re
from pathlib import Path
from datasets import load_dataset
from IPython.display import Video, display
from safetensors.torch import load_file

from src.config import Config
from src.model.virality_predictor import ViralityPredictor
from src.model.data_processor import DataProcessor

device = torch.device('mps' if torch.backends.mps.is_available(
) else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

config = Config()
checkpoint_dir = Path(config.checkpoint_path)
checkpoint_folders = [f for f in checkpoint_dir.iterdir(
) if f.is_dir() and re.match(r'checkpoint-\d+$', f.name)]
latest_checkpoint = max(
    checkpoint_folders, key=lambda f: int(f.name.split('-')[1]))
print(f'Loading model from: {latest_checkpoint}')

model = ViralityPredictor(config)
state_dict = load_file(latest_checkpoint / 'model.safetensors')
model.load_state_dict(state_dict)
model.to(device)
model.eval()

assert model.tabular_means is not None and model.tabular_stds is not None

processor = DataProcessor(config)
processor.tabular_means = torch.tensor(model.tabular_means.cpu())
processor.tabular_stds = torch.tensor(model.tabular_stds.cpu())

dataset = load_dataset(config.dataset_id)['train']
example_idx = np.random.randint(0, len(dataset))
example = dataset[example_idx]

print(f'\n=== Ground Truth (Example {example_idx}) ===')
print(f'Engagement Score: {example["engagement_score"]:.2f}')
print(f'Velocity Score:   {example["view_velocity_score"]:.2f}')
print(f'Is Viral:         {bool(example["is_viral"])}')

batch = {k: [v] for k, v in example.items()}
processed = processor._process_batch(batch)
inference_batch = {k: v.to(device)
                   for k, v in processed.items() if k != 'labels'}

predictions = model.predict_scores(**inference_batch)

prob = predictions["viral_prob"].item()
PRECISION_THRESHOLD = 0.7

print(f'\n=== Model Predictions (Inverse Log Space) ===')
print(f'Pred Engagement: {predictions["engagement"].item():.2f}')
print(f'Pred Velocity:   {predictions["velocity"].item():.2f}')
print(f'Viral Confidence: {prob:.2%}')
print(
    f'Classification:  {"VIRAL" if prob >= PRECISION_THRESHOLD else "NOT VIRAL"}')

if example['video_bytes']:
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as tmp:
        tmp.write(example['video_bytes'])
        display(Video(tmp.name, embed=True, width=300))